In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

import torch
import numpy as np
from torch.utils.data import ConcatDataset, Subset
from sklearn.model_selection import StratifiedKFold
import torchvision.transforms as transforms

# Import using package structure to avoid relative import issues
from benchmarks.medMNIST.utils import train_models_load_datasets as tr

print(f"PyTorch version: {torch.__version__}")
print(f"Working directory: {Path.cwd()}")
print(f"Project root: {project_root}")

## Check for Data Leakage in OrganAMNIST

This notebook verifies whether the calibration set overlaps with training data used in any CV fold.

In [ ]:
# Configuration
flag = 'organamnist'
image_size = 224
batch_size = 128
n_splits = 5
seed = 42

print(f"Dataset: {flag}")
print(f"CV splits: {n_splits}")
print(f"Seed: {seed}")

## 1. Load and Split Data (Same as Training Pipeline)

In [ ]:
# RepeatGrayToRGB transform
class RepeatGrayToRGB:
    def __call__(self, x):
        return x.repeat(3, 1, 1)

color = False
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5]),
    RepeatGrayToRGB(),
])

# Load datasets using the SAME function as training
[study_dataset, calib_dataset, test_dataset], \
[_, calib_loader, test_loader], info = \
    tr.load_datasets(flag, color, image_size, transform, batch_size)

print(f"\nDataset sizes:")
print(f"  Study (train): {len(study_dataset)}")
print(f"  Calibration: {len(calib_dataset)}")
print(f"  Test: {len(test_dataset)}")

## 2. Get Calibration Indices

In [ ]:
# Calibration dataset is a Subset - get its indices
if isinstance(calib_dataset, Subset):
    calib_indices = set(calib_dataset.indices)
    print(f"Calibration indices (first 20): {sorted(list(calib_indices))[:20]}")
    print(f"Total calibration samples: {len(calib_indices)}")
else:
    print("ERROR: calib_dataset is not a Subset!")
    calib_indices = set()

## 3. Simulate CV Splits (Same as Training)

In [ ]:
# Get the base combined dataset (train + val from medMNIST)
if isinstance(study_dataset, Subset):
    combined_dataset = study_dataset.dataset
    study_indices = study_dataset.indices
    print(f"Study dataset is a Subset with {len(study_indices)} indices")
else:
    # If it's already the combined dataset
    combined_dataset = study_dataset
    study_indices = list(range(len(study_dataset)))
    print(f"Study dataset is the full dataset with {len(study_indices)} samples")

# Extract labels for stratification (same as CV_fold_generator)
labels = []
for idx in study_indices:
    _, label = combined_dataset[idx]
    if isinstance(label, torch.Tensor):
        label = label.item()
    elif isinstance(label, np.ndarray):
        label = label.item() if label.size == 1 else label[0]
    labels.append(label)

print(f"\nExtracted {len(labels)} labels for stratification")
print(f"Unique labels: {sorted(set(labels))}")

## 4. Check Overlap Between CV Folds and Calibration Set

In [ ]:
# Perform stratified K-fold split (SAME as training)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

print("\nChecking for data leakage across CV folds:")
print("=" * 80)

fold_train_indices = []
total_overlap_samples = set()

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(study_indices, labels)):
    # Convert to actual dataset indices
    train_fold_indices = [study_indices[i] for i in train_idx]
    val_fold_indices = [study_indices[i] for i in val_idx]
    
    # Check overlap between this fold's training data and calibration set
    train_set = set(train_fold_indices)
    overlap = train_set.intersection(calib_indices)
    
    fold_train_indices.append(train_set)
    total_overlap_samples.update(overlap)
    
    print(f"\nFold {fold_idx}:")
    print(f"  Training samples: {len(train_fold_indices)}")
    print(f"  Validation samples: {len(val_fold_indices)}")
    print(f"  Overlap with calibration: {len(overlap)} samples")
    
    if len(overlap) > 0:
        print(f"  ⚠️  LEAKAGE DETECTED! {len(overlap)}/{len(calib_indices)} "
              f"({len(overlap)/len(calib_indices)*100:.1f}%) calibration samples seen in training")

print("\n" + "=" * 80)
print(f"\nSUMMARY:")
print(f"Total calibration samples: {len(calib_indices)}")
print(f"Calibration samples seen in ANY fold: {len(total_overlap_samples)}")
print(f"Leakage percentage: {len(total_overlap_samples)/len(calib_indices)*100:.1f}%")

if len(total_overlap_samples) > 0:
    print(f"\n❌ DATA LEAKAGE CONFIRMED!")
    print(f"   {len(total_overlap_samples)} calibration samples were used for training")
else:
    print(f"\n✅ NO DATA LEAKAGE: Calibration set is completely separate")

## 5. Check if Calibration Samples Appear in Multiple Folds

In [ ]:
# For each calibration sample, count how many folds it appears in
if len(total_overlap_samples) > 0:
    print("\nAnalyzing how many folds each leaked sample appears in:")
    print("=" * 80)
    
    leak_counts = {}
    for calib_idx in total_overlap_samples:
        count = sum(1 for fold_set in fold_train_indices if calib_idx in fold_set)
        leak_counts[calib_idx] = count
    
    # Distribution of leak counts
    count_distribution = {}
    for count in leak_counts.values():
        count_distribution[count] = count_distribution.get(count, 0) + 1
    
    print(f"\nDistribution of leaked samples across folds:")
    for num_folds in sorted(count_distribution.keys()):
        num_samples = count_distribution[num_folds]
        print(f"  Appeared in {num_folds}/{n_splits} folds: {num_samples} samples "
              f"({num_samples/len(total_overlap_samples)*100:.1f}%)")
    
    print(f"\nConclusion:")
    print(f"  - Each leaked sample appears in ~{np.mean(list(leak_counts.values())):.1f} folds on average")
    print(f"  - This explains why ensemble performance is so good on calibration set!")

## 6. Verify Index Mapping

In [ ]:
# Double-check the index mapping is correct
print("\nVerifying index mappings:")
print("=" * 80)

# Get the actual combined dataset (before 80/20 split)
datasets_raw, _ = tr.get_datasets(flag, im_size=image_size, color=color, transform=transform)
train_raw = datasets_raw[0]
val_raw = datasets_raw[1]

print(f"Raw medMNIST splits:")
print(f"  Train: {len(train_raw)}")
print(f"  Val: {len(val_raw)}")
print(f"  Combined (train+val): {len(train_raw) + len(val_raw)}")

# Check if the combined size matches
if isinstance(study_dataset, Subset):
    print(f"\nStudy dataset (Subset):")
    print(f"  Base dataset size: {len(study_dataset.dataset)}")
    print(f"  Subset size: {len(study_dataset)}")
    
if isinstance(calib_dataset, Subset):
    print(f"\nCalibration dataset (Subset):")
    print(f"  Base dataset size: {len(calib_dataset.dataset)}")
    print(f"  Subset size: {len(calib_dataset)}")
    
    # Check if they share the same base
    if isinstance(study_dataset, Subset):
        same_base = (study_dataset.dataset is calib_dataset.dataset)
        print(f"\n  Same base dataset: {same_base}")
        
        if same_base:
            print(f"  ✅ Both subsets come from the same ConcatDataset (train+val)")
            print(f"     Combined size: {len(study_dataset) + len(calib_dataset)}")
            print(f"     Expected: {len(train_raw) + len(val_raw)}")